# **Homework 2: Phoneme Classification**


Objectives:
* Solve a classification problem with deep neural networks (DNNs).
* Understand recursive neural networks (RNNs).

If you have any questions, please contact the TAs via TA hours, NTU COOL, or email to mlta-2023-spring@googlegroups.com

# Some Utility Functions
**Fixes random number generator seeds for reproducibility.**

In [1]:
import numpy as np
import torch
import random

def same_seeds(seed):
    random.seed(seed) 
    np.random.seed(seed)  
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) 
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

**Helper functions to pre-process the training data from raw MFCC features of each utterance.**

A phoneme may span several frames and is dependent to past and future frames. \
Hence we concatenate neighboring phonemes for training to achieve higher accuracy. The **concat_feat** function concatenates past and future k frames (total 2k+1 = n frames), and we predict the center frame.

Feel free to modify the data preprocess functions, but **do not drop any frame** (if you modify the functions, remember to check that the number of frames are the same as mentioned in the slides)

In [2]:
import os
import torch
from tqdm import tqdm

mask_len = 5
thre = 0.03

def load_feat(path):
    feat = torch.load(path)
    feat = (feat - feat.mean(dim=0, keepdim=True)) / (feat.std(dim=0, keepdim=True) + 1E-8)
    return feat

def shift(x, n):
    if n < 0:
        left = x[0].repeat(-n, 1)
        right = x[:n]
    elif n > 0:
        right = x[-1].repeat(n, 1)
        left = x[n:]
    else:
        return x

    return torch.cat((left, right), dim=0)

def concat_feat(x, concat_n):
    assert concat_n % 2 == 1 # n must be odd
    if concat_n < 2:
        return x
    seq_len, feature_dim = x.size(0), x.size(1)
    x = x.repeat(1, concat_n) 
    x = x.view(seq_len, concat_n, feature_dim).permute(1, 0, 2) # concat_n, seq_len, feature_dim
    mid = (concat_n // 2)
    for r_idx in range(1, mid+1):
        x[mid + r_idx, :] = shift(x[mid + r_idx], r_idx)
        x[mid - r_idx, :] = shift(x[mid - r_idx], -r_idx)

    return x.permute(1, 0, 2).view(seq_len, concat_n * feature_dim)

def preprocess_data(split, feat_dir, phone_path, concat_nframes, train_ratio=0.8):
    class_num = 41 # NOTE: pre-computed, should not need change

    if split == 'train' or split == 'val':
        mode = 'train'
    elif split == 'test':
        mode = 'test'
    else:
        raise ValueError('Invalid \'split\' argument for dataset: PhoneDataset!')

    label_dict = {}
    if mode == 'train':
        for line in open(os.path.join(phone_path, f'{mode}_labels.txt')).readlines():
            line = line.strip('\n').split(' ')
            label_dict[line[0]] = [int(p) for p in line[1:]]
        
        # split training and validation data
        usage_list = open(os.path.join(phone_path, 'train_split.txt')).readlines()
        random.shuffle(usage_list)
        train_len = int(len(usage_list) * train_ratio)
        usage_list = usage_list[:train_len] if split == 'train' else usage_list[train_len:]

    elif mode == 'test':
        usage_list = open(os.path.join(phone_path, 'test_split.txt')).readlines()

    usage_list = [line.strip('\n') for line in usage_list]
    print('[Dataset] - # phone classes: ' + str(class_num) + ', number of utterances for ' + split + ': ' + str(len(usage_list)))

    max_len = 3000000
    X = torch.empty(max_len, 39 * concat_nframes)
    if mode == 'train':
        y = torch.empty(max_len, dtype=torch.long)

    idx = 0
    for i, fname in tqdm(enumerate(usage_list)):
        feat = load_feat(os.path.join(feat_dir, mode, f'{fname}.pt'))
        if mode == 'train':
            # noise
            feat += 0.005 * torch.randn_like(feat)
            # rnd msk
            start = random.randint(0, len(feat) - mask_len)
            feat[start:start + mask_len] = 0
            # rnd dropout
            mask = (torch.rand(feat.size(0), 1) > thre).float()
            feat *= mask

        cur_len = len(feat)
        feat = concat_feat(feat, concat_nframes)
        if mode == 'train':
          label = torch.LongTensor(label_dict[fname])

        X[idx: idx + cur_len, :] = feat
        if mode == 'train':
          y[idx: idx + cur_len] = label

        idx += cur_len

    X = X[:idx, :]
    if mode == 'train':
      y = y[:idx]

    print(f'[INFO] {split} set')
    print(X.shape)
    if mode == 'train':
      print(y.shape)
      return X, y
    else:
      return X


# Dataset

In [3]:
import torch
from torch.utils.data import Dataset

class LibriDataset(Dataset):
    def __init__(self, X, y=None):
        self.data = X
        if y is not None:
            self.label = torch.LongTensor(y)
        else:
            self.label = None

    def __getitem__(self, idx):
        if self.label is not None:
            return self.data[idx], self.label[idx]
        else:
            return self.data[idx]

    def __len__(self):
        return len(self.data)


# Hyper-parameters

In [4]:
# data prarameters
concat_nframes = 23              # the number of frames to concat with, n must be odd (total 2k+1 = n frames)
train_ratio = 0.80               # the ratio of data used for training, the rest will be used for validation

# training parameters
seed = 42                        # random seed
batch_size = 512                # batch size
num_epoch = 30                   # the number of training epoch
early_stop = num_epoch // 4
learning_rate = 1e-4         # learning rate
model_path = './model.ckpt'     # the path where the checkpoint will be saved

# model parameters
input_dim = 39 * concat_nframes # the input dim of the model, you should not change the value
hidden_layers = 3               # the number of hidden layers
hidden_dim = 256                # the hidden dim

# Model
Feel free to modify the structure of the model.

In [5]:
import torch.nn as nn

class Classifier(nn.Module):
    def __init__(self, batch_size=batch_size, num_layers=hidden_layers, hidden_dim=hidden_dim, seq_length=concat_nframes):
        super(Classifier, self).__init__()
        self.batch_size = batch_size
        self.num_layers = hidden_layers
        self.hidden_dim = hidden_dim
        self.seq_length = seq_length
        self.lstm = nn.LSTM(input_size=39, hidden_size=hidden_dim, num_layers=hidden_layers,\
                           batch_first=True, dropout=0.5, bidirectional=True)
        self.fc = nn.Sequential(
          nn.LeakyReLU(),
          nn.BatchNorm1d(2 * hidden_dim),
          nn.Dropout(0.5),
          nn.Linear(2 * hidden_dim, hidden_dim),
          nn.LeakyReLU(),
          nn.BatchNorm1d(hidden_dim),
          nn.Dropout(0.5),
          nn.Linear(hidden_dim, 41))
    def forward(self, x):
        x, _ = self.lstm(x)
        x = x[:, self.seq_length // 2]
        x = self.fc(x)
        return x

In [6]:
# 使用标签平滑正则化
# Ref: https://github.com/wangleiofficial/label-smoothing-pytorch/tree/main
import torch.nn.functional as F

def linear_combination(x, y, epsilon):
    return epsilon * x + (1 - epsilon) * y


def reduce_loss(loss, reduction='mean'):
    return loss.mean() if reduction == 'mean' else loss.sum() if reduction == 'sum' else loss


class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, epsilon: float = 0.1, reduction='mean'):
        super().__init__()
        self.epsilon = epsilon
        self.reduction = reduction

    def forward(self, preds, target):
        n = preds.size()[-1]
        log_preds = F.log_softmax(preds, dim=-1)
        loss = reduce_loss(-log_preds.sum(dim=-1), self.reduction)
        nll = F.nll_loss(log_preds, target, reduction=self.reduction)
        return linear_combination(loss / n, nll, self.epsilon)


# Dataloader

In [7]:
from torch.utils.data import DataLoader
import gc

same_seeds(seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {device}')

# preprocess data
train_X, train_y = preprocess_data(split='train', feat_dir='/kaggle/input/ml2023spring-hw2/libriphone/feat', phone_path='/kaggle/input/ml2023spring-hw2/libriphone', concat_nframes=concat_nframes, train_ratio=train_ratio)
val_X, val_y = preprocess_data(split='val', feat_dir='/kaggle/input/ml2023spring-hw2/libriphone/feat', phone_path='/kaggle/input/ml2023spring-hw2/libriphone', concat_nframes=concat_nframes, train_ratio=train_ratio)

# get dataset
train_set = LibriDataset(train_X, train_y)
val_set = LibriDataset(val_X, val_y)

# remove raw feature to save memory
del train_X, train_y, val_X, val_y
gc.collect()

# get dataloader
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

DEVICE: cuda
[Dataset] - # phone classes: 41, number of utterances for train: 2743


2743it [00:19, 140.05it/s]


[INFO] train set
torch.Size([1689304, 897])
torch.Size([1689304])
[Dataset] - # phone classes: 41, number of utterances for val: 686


686it [00:02, 234.11it/s]


[INFO] val set
torch.Size([423765, 897])
torch.Size([423765])


# Training

In [8]:
# create model, define a loss function, and optimizer
model = Classifier(batch_size, hidden_layers, hidden_dim, concat_nframes).to(device)
# Define a loss function, and optimizer
criterion = LabelSmoothingCrossEntropy(reduction='sum')
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
# Create a learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=2, T_mult=4, eta_min=0)
# for early stop
early_count = 0

best_acc = 0.0
for epoch in range(num_epoch):
    train_acc = 0.0
    train_loss = 0.0
    val_acc = 0.0
    val_loss = 0.0
    
    # training
    model.train() # set the model to training mode
    for i, batch in enumerate(tqdm(train_loader)):
        features, labels = batch
        features = features.view(-1, concat_nframes, 39).to(device) # feature.shape: (batch_size, seq_len, input_size)
        labels = labels.to(device)
        
        optimizer.zero_grad() 
        outputs = model(features) # (batch_size, labels) 
        
        loss = criterion(outputs, labels)
        loss.backward() 
        optimizer.step() 
        
        _, train_pred = torch.max(outputs, 1) # get the index of the class with the highest probability
        train_acc += (train_pred.detach() == labels.detach()).sum().item()
        train_loss += loss.item()
    scheduler.step()

    # validation
    model.eval() # set the model to evaluation mode
    with torch.no_grad():
        for i, batch in enumerate(tqdm(val_loader)):
            features, labels = batch
            features = features.view(-1, concat_nframes, 39).to(device)
            labels = labels.to(device)
            outputs = model(features)
            
            loss = criterion(outputs, labels) 
            
            _, val_pred = torch.max(outputs, 1) 
            val_acc += (val_pred.cpu() == labels.cpu()).sum().item() # get the index of the class with the highest probability
            val_loss += loss.item()

    print(f'[{epoch+1:03d}/{num_epoch:03d}] Train Acc: {train_acc/len(train_set):3.5f} Loss: {train_loss/len(train_loader):3.5f} | Val Acc: {val_acc/len(val_set):3.5f} loss: {val_loss/len(val_loader):3.5f}')

    # if the model improves, save a checkpoint at this epoch
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), model_path)
        print(f'saving model with acc {best_acc/len(val_set):.5f}')
        early_count = 0
    else:
        early_count += 1
        if early_count >= early_stop:
            print('\nModel is not improving, so we halt the training session.')
            break
            
print(f'saving model with acc {best_acc/len(val_set):.5f}')

100%|██████████| 828/828 [00:15<00:00, 53.01it/s]


[001/030] Train Acc: 0.51893 Loss: 1062.64913 | Val Acc: 0.64568 loss: 843.77468
saving model with acc 0.64568


100%|██████████| 828/828 [00:15<00:00, 53.06it/s]


[002/030] Train Acc: 0.64329 Loss: 866.37958 | Val Acc: 0.68899 loss: 780.38893
saving model with acc 0.68899


100%|██████████| 828/828 [00:15<00:00, 53.50it/s]


...

[028/030] Train Acc: 0.90803 Loss: 493.35032 | Val Acc: 0.88418 loss: 530.83948
saving model with acc 0.88418


100%|██████████| 828/828 [00:15<00:00, 53.49it/s]


[029/030] Train Acc: 0.91025 Loss: 490.13517 | Val Acc: 0.88619 loss: 529.29262
saving model with acc 0.88619


100%|██████████| 828/828 [00:15<00:00, 52.98it/s]

[030/030] Train Acc: 0.91257 Loss: 487.16016 | Val Acc: 0.88775 loss: 527.95270
saving model with acc 0.88775
saving model with acc 0.88775


In [9]:
del train_set, val_set
del train_loader, val_loader
gc.collect()

21

# Testing
Create a testing dataset, and load model from the saved checkpoint.

In [10]:
# load data
test_X = preprocess_data(split='test', feat_dir='/kaggle/input/ml2023spring-hw2/libriphone/feat', phone_path='/kaggle/input/ml2023spring-hw2/libriphone', concat_nframes=concat_nframes)
test_set = LibriDataset(test_X, None)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

[Dataset] - # phone classes: 41, number of utterances for test: 857


857it [00:08, 96.06it/s]

[INFO] test set
torch.Size([527364, 897])


In [11]:
# load model
model = Classifier(batch_size, hidden_layers, hidden_dim, concat_nframes).to(device)
model.load_state_dict(torch.load(model_path))

<All keys matched successfully>

Make prediction.

In [12]:
pred = np.array([], dtype=np.int32)

model.eval()
with torch.no_grad():
    for i, batch in enumerate(tqdm(test_loader)):
        features = batch
        features = features.view(-1, concat_nframes, 39).to(device)
        outputs = model(features)

        _, test_pred = torch.max(outputs, 1) # get the index of the class with the highest probability
        pred = np.concatenate((pred, test_pred.cpu().numpy()), axis=0)
        

100%|██████████| 1031/1031 [00:18<00:00, 57.16it/s]


Write prediction to a CSV file.

After finish running this block, download the file `prediction.csv` from the files section on the left-hand side and submit it to Kaggle.

In [13]:
with open('prediction.csv', 'w') as f:
    f.write('Id,Class\n')
    for i, y in enumerate(pred):
        f.write('{},{}\n'.format(i, y))